### Importações para Code Review RAG

Bibliotecas especializadas para análise de código:

- **GenericLoader + LanguageParser**: Carregam código-fonte Python preservando estrutura semântica (não apenas texto)
- **Language enum**: Especifica que estamos processando código Python (afeta chunking e parsing)
- **ChatPromptTemplate**: Cria prompts estruturados com roles (system, user) para melhor controle
- **create_retrieval_chain**: Orquestra recuperação de documentos + geração de resposta
- **create_stuff_documents_chain**: Insere documentos no contexto do LLM (em vez de map-reduce)
- **Repo (GitPython)**: Clona e gerencia repositórios Git localmente

In [ ]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
import os
from git import Repo

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

### Clonagem do Repositório

Estamos analisando o **LangChain Core** (a biblioteca Python da LangChain):
- `repo_path='./repo_review'`: Cria um diretório local com todo o repositório
- `Repo.clone_from()`: Git clone da URL pública
- Este é um repositório real com centenas de arquivos Python prontos para análise

In [6]:
repo_path = './repo_review'

In [8]:
repo = Repo.clone_from('https://github.com/langchain-ai/langchain', to_path=repo_path)

### Carregamento Inteligente de Código Python

**GenericLoader** com **LanguageParser**:
- Carrega todos os arquivos `.py` do diretório `langchain_core`
- `exclude`: Ignora arquivos quebrados (encoding UTF-8 incorreto)
- **LanguageParser**: Parser especializado que entende estrutura Python
  - Divide por classes, funções, imports (não apenas quebras de linha)
  - `parser_threshold=500`: Ignora arquivos muito pequenos
- `glob='**/*'`: Busca recursivamente em todas as subpastas

Resultado: **641 documentos** = arquivos ou blocos de código extraído do repositório

In [9]:
loader = GenericLoader.from_filesystem(
    os.path.join(repo_path, 'libs/core/langchain_core'),
    glob='**/*',
    suffixes=['.py'],
    exclude=['**/non-utf-8-enconding.py'],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

documents = loader.load()

In [10]:
len(documents)

641

### Splitting Especializado para Python

**RecursiveCharacterTextSplitter.from_language(Language.PYTHON)**:
- Divide respeitando a sintaxe Python
- Quebra por: funções, classes, blocos de código (não no meio de uma definição)
- `chunk_size=2_000`: Chunks menores (400 linhas típicas de código Python)
- `chunk_overlap=200`: 200 caracteres de sobreposição (preserva contexto de imports/definições)

Resultado: **1803 chunks** (641 docs → 1803 chunks)
- Aumentou 2.8x porque o splitting descompactou o código em blocos lógicos
- Cada chunk é uma unidade semântica completa (ex: uma função inteira)

In [11]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=2_000,
    chunk_overlap=200
)

texts = python_splitter.split_documents(documents)

In [12]:
len(texts)

1803

### Vector Store com Busca MMR

**Chroma** armazena embeddings de código:
- Cada chunk Python é convertido em vetor usando OpenAIEmbeddings
- `disallowed_special=()`: Permite caracteres especiais do Python (decoradores, operadores, etc)

**Retriever com MMR (Maximal Marginal Relevance)**:
- `search_type='mmr'`: Em vez de similaridade simples, busca documentos relevantes E diversos
- `search_kwargs={'k': 8}`: Recupera os 8 chunks mais relevantes
- **Por que MMR?** Para análise de código, queremos exemplos diversos, não repetidos
  - Evita retornar 8 linhas similares do mesmo padrão
  - Maximiza cobertura de contexto para o revisor

In [ ]:
db = Chroma.from_documents(texts, OpenAIEmbeddings(disallowed_special=()))

retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 8},
)

### Configuração do Modelo LLM

**ChatOpenAI** com parâmetros otimizados para code review:
- `model='gpt-3.5-turbo'`: Modelo rápido e eficiente em custo
- `max_tokens=1_000`: Limite maior (vs 200 no RAG de documentos)
  - Code review precisa de respostas mais detalhadas
  - 1000 tokens ≈ 750 palavras, espaço para múltiplas recomendações

In [20]:
llm = ChatOpenAI(model='gpt-3.5-turbo', max_tokens=1_000)

### Prompt Estruturado para Code Review

**ChatPromptTemplate** define dois roles:

**Sistema**:
- Define o papel: "Você é um revisor de código Python"
- Especifica 4 categorias de análise:
  1. **Bugs/Erros**: Lógica, exceções não tratadas
  2. **Performance**: Loops, operações custosas, estruturas ineficientes
  3. **Qualidade**: Nomes, duplicação, complexidade
  4. **Python idiomático**: Type hints, comprehensions, context managers
- Instrui formato: "problema → explicação breve → sugestão"
- `{context}`: Placeholder para documentos recuperados (código relevante)

**Usuário**:
- `{input}`: Placeholder para a pergunta/requisição do código a revisar

Este formato estruturado melhora a qualidade das respostas do LLM

In [21]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '''Você é um revisor de código Python. Analise o código e aponte:

1. **Bugs/Erros**: lógica incorreta, exceções não tratadas
2. **Performance**: loops ineficientes, operações custosas, uso de estruturas
3. **Qualidade**: nomes confusos, duplicação, complexidade alta
4. **Python idiomático**: use de type hints, list comprehensions, context managers

Seja direto. Format: problema → explicação breve → sugestão.

Contexto:
{context}'''
        ),
        (
            'user',
            '{input}'
        )
    ]
)


### Construção da Chain RAG Completa

**create_stuff_documents_chain(llm, prompt)**:
- Combina documentos recuperados + prompt → cria uma chain
- "stuff": Coloca todos os documentos diretamente no contexto do LLM
- Retorna: uma chain que processa `{context}` e `{input}`

**create_retrieval_chain(retriever, document_chain)**:
- Orquestra o pipeline RAG completo:
  1. Recebe `{input}` (pergunta do usuário)
  2. Executa `retriever.invoke(input)` → busca documentos relevantes
  3. Passa documentos para `document_chain` como `{context}`
  4. Retorna resposta final
- Retorna um dicionário com chave `'answer'`

Esta é a cadeia **RAG end-to-end**: Retrieve → Augment → Generate

In [22]:
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever, document_chain)

### Executando o Code Review RAG

**Fluxo**:
1. Input: `'Você pode revisar sugerir melhorias para o código de RunnableBinding'`
2. Retriever busca 8 chunks relevantes sobre `RunnableBinding` no LangChain Core
3. Esses chunks + prompt → enviados para o LLM
4. LLM analisa o código contra as 4 categorias (Bugs, Performance, Qualidade, Python idiomático)
5. Retorna revisão estruturada com problemas e sugestões

**Resultado esperado**:
- `response['answer']`: String com a análise de code review completa
- Baseado em **código real** do repositório (RAG), não apenas conhecimento treinado

In [24]:
response = retrieval_chain.invoke({
    'input': 'Você pode revisar sugerir melhorias para o código de RunnableBinding'
})

In [25]:
print(response['answer'])

1. **Bugs/Erros**:
   - A classe `RunnableBinding` está definida duas vezes no código, o que pode causar confusão e erros de importação.
  
### Sugestão para Bugs/Erros:
Remova a segunda definição da classe `RunnableBinding`.

2. **Performance**:
   - O código contém trechos extensos de documentação que podem tornar a leitura do código mais difícil, especialmente em funções simples.
   - O uso extensivo de anotações de tipo pode tornar o código mais verboso.
  
### Sugestão de Performance:
Escreva documentação mais concisa e evite anotações de tipo excessivas onde não necessário.

3. **Qualidade**:
   - Os nomes de algumas funções e classes poderiam ser mais descritivos.
   - Alguns trechos de código escondem a lógica real de implementação, o que pode dificultar a compreensão.

### Sugestão de Qualidade:
   - Use nomes mais descritivos para funções e variáveis.
   - Evite esconder a lógica por trás de muitas camadas de abstração.

4. **Python idiomático**:
   - Utilize o método `random